<a href="https://colab.research.google.com/github/HariniAnandkumar/neuromorphic-ai-research/blob/main/01_ann_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: Tesla T4


In [ ]:
!pip install thop torchinfo -q

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import pandas as pd
from thop import profile
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
transform = transforms.ToTensor()

trainset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform)

testset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 501kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.78MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.12MB/s]


In [ ]:
class ANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, 3),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Flatten(),
            nn.Linear(9216, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x)

model = ANN().to(device)
summary(model, (1, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
ANN                                      [1, 10]                   --
├─Sequential: 1-1                        [1, 10]                   --
│    └─Conv2d: 2-1                       [1, 32, 26, 26]           320
│    └─ReLU: 2-2                         [1, 32, 26, 26]           --
│    └─Conv2d: 2-3                       [1, 64, 24, 24]           18,496
│    └─ReLU: 2-4                         [1, 64, 24, 24]           --
│    └─MaxPool2d: 2-5                    [1, 64, 12, 12]           --
│    └─Dropout: 2-6                      [1, 64, 12, 12]           --
│    └─Flatten: 2-7                      [1, 9216]                 --
│    └─Linear: 2-8                       [1, 128]                  1,179,776
│    └─ReLU: 2-9                         [1, 128]                  --
│    └─Linear: 2-10                      [1, 10]                   1,290
Total params: 1,199,882
Trainable params: 1,199,882
Non-trainable para

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
epochs = 10
start_time = time.time()

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(trainloader):.4f}")

train_time = time.time() - start_time
print("Training Time (sec):", train_time)

Epoch 1/10 | Loss: 0.1925
Epoch 2/10 | Loss: 0.0517
Epoch 3/10 | Loss: 0.0347
Epoch 4/10 | Loss: 0.0254
Epoch 5/10 | Loss: 0.0187
Epoch 6/10 | Loss: 0.0142
Epoch 7/10 | Loss: 0.0121
Epoch 8/10 | Loss: 0.0120
Epoch 9/10 | Loss: 0.0097
Epoch 10/10 | Loss: 0.0080
Training Time (sec): 105.7238667011261


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print("Test Accuracy:", test_acc)

Test Accuracy: 98.86


In [ ]:
dummy_input = torch.randn(1, 1, 28, 28).to(device)
flops, params = profile(model, inputs=(dummy_input,))
print("FLOPs:", flops)
print("Params:", params)

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
FLOPs: 11992448.0
Params: 1199882.0


In [ ]:
metrics = {
    "model": "ANN",
    "accuracy": test_acc,
    "train_time_sec": train_time,
    "params": params,
    "flops": flops,
    "spike_count": 0
}

df = pd.DataFrame([metrics])
df.to_csv("ann_mnist_results.csv", index=False)

print("Saved ann_mnist_results.csv")

Saved ann_mnist_results.csv
